# Machine Learning-Based Prediction of 30-Day Hospital Readmission for Diabetic Patients

This notebook covers the exploratory data analysis (EDA), preprocessing, and modeling phases of the project. We will:
1. **Load and Explore the Dataset**: Understand the structure, shape, and types of data.
2. **Feature Explanation**: Define each clinical column in plain, simple terms.
3. **Data Preprocessing**: Handle missing values, filter out deceased/hospice patients, deduplicate patient encounters, and apply SMOTE to handle class imbalance.
4. **Model Training & Comparison**: Train Logistic Regression, Decision Trees, Random Forest, and XGBoost.
5. **Model Evaluation**: Analyze accuracy, precision, recall, F1, and ROC-AUC curves.
6. **Model Interpretation**: Use SHAP to explain how features influence patient risk.

## 1. Import Libraries
First, we import the core scientific, visualization, and machine learning libraries.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, roc_curve
)
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully!")

## 2. Load the Dataset and Feature Explanations
We load the raw dataset (`diabetic_data.csv`) and detail the meanings of the columns.

In [ ]:
# Load raw dataset
csv_path = "../dataset/raw/dataset_diabetes/diabetic_data.csv"
if not os.path.exists(csv_path):
    csv_path = "../dataset/raw/diabetic_data.csv"
    
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
else:
    print("Dataset file not found! Please run the download script first.")

### Feature Explanations (Key Columns)
- `encounter_id`: A unique identifier for each hospital encounter (admission).
- `patient_nbr`: A unique identification number for each patient.
- `race`: Nominal category describing ethnicity (Caucasian, African American, Hispanic, Asian, etc.).
- `gender`: Patient's biological sex (Male or Female).
- `age`: Grouped in 10-year intervals (e.g., `[0-10)`, `[10-20)`, etc.).
- `weight`: Patient weight in pounds (highly missing in this dataset).
- `admission_type_id`: Integer representing emergency, elective, urgent, newborn, etc.
- `discharge_disposition_id`: Integer representing where the patient went after discharge (Home, another facility, hospice, or expired).
- `admission_source_id`: Integer representing where the patient was admitted from (Emergency Room, physician referral, transfer).
- `time_in_hospital`: Numeric length of hospital stay in days.
- `payer_code`: Payer class (Medicare, Medicaid, Private insurance, etc.).
- `medical_specialty`: Specialty of the admitting physician (Cardiology, Internal Medicine, etc.).
- `num_lab_procedures`: Number of lab tests performed during the encounter.
- `num_procedures`: Number of procedures other than lab tests performed during the encounter.
- `num_medications`: Number of unique generic medications administered during the encounter.
- `number_outpatient`: Number of outpatient visits in the year preceding the encounter.
- `number_emergency`: Number of emergency room visits in the year preceding the encounter.
- `number_inpatient`: Number of inpatient admissions in the year preceding the encounter.
- `diag_1`, `diag_2`, `diag_3`: Primary, secondary, and tertiary diagnosis codes (ICD-9 classification).
- `number_diagnoses`: Number of diagnoses entered into the system.
- `max_glu_serum`: Results of a glucose serum test (">200", ">300", "normal", "none").
- `A1Cresult`: Results of an HbA1c test (">7", ">8", "normal", "none").
- `change`: Indicates if there was a dosage change in any of the patient's diabetic medications (Ch / No).
- `diabetesMed`: Indicates if any diabetic medication was prescribed to the patient (Yes / No).
- `readmitted`: Target variable detailing if the patient was readmitted. `<30` (readmitted in <30 days), `>30` (readmitted in >30 days), or `NO` (no readmission).

## 3. Exploratory Data Analysis (EDA)
We investigate structural stats, missing values, duplicates, and visual distributions.

In [ ]:
# Display basic info
print("=== Data Types and Non-Null Counts ===")
print(df.info())

# Check for missing values encoded as '?'
print("\n=== Missing values count (represented as '?') ===")
for col in df.columns:
    nulls = (df[col] == '?').sum()
    if nulls > 0:
        pct = (nulls / len(df)) * 100
        print(f"{col}: {nulls} ({pct:.2f}% missing)")

In [ ]:
# Display summary statistics for numeric variables
print("=== Descriptive Statistics for Numeric Columns ===")
df.describe().T

In [ ]:
# Set visual aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# 1. Plot Class Imbalance in Target Variable
plt.figure(figsize=(6, 4))
sns.countplot(x='readmitted', data=df, palette='viridis')
plt.title('Distribution of Readmission in Raw Data')
plt.show()

# 2. Plot Distribution of Time In Hospital
plt.figure(figsize=(8, 4))
sns.histplot(df['time_in_hospital'], bins=14, kde=True, color='teal')
plt.title('Distribution of Length of Hospital Stay (Days)')
plt.xlabel('Days')
plt.show()

## 4. Data Preprocessing
We write helper functions to format the columns and transform the dataframe, following the logic of `src/preprocess.py`.

In [ ]:
# Replace missing indicator '?' with NaN
df_clean = df.replace('?', np.nan)

# Filter out dead or hospice-discharged patients
dead_hospice_ids = [11, 13, 14, 19, 20, 21, 25]
df_clean = df_clean[~df_clean['discharge_disposition_id'].isin(dead_hospice_ids)]

# Keep only first encounter per patient
df_clean = df_clean.sort_values('encounter_id')
df_clean = df_clean.drop_duplicates(subset='patient_nbr', keep='first')

# Map target variable to binary (1 for <30 days, 0 for others)
df_clean['readmitted'] = df_clean['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

# Map age categories to midpoints
def clean_age(age_str):
    try:
        age_str = age_str.replace('[', '').replace(')', '')
        start, end = map(int, age_str.split('-'))
        return (start + end) / 2
    except:
        return 50

df_clean['age'] = df_clean['age'].apply(clean_age)

# Drop uninformative columns
df_clean = df_clean.drop(columns=['weight', 'payer_code', 'encounter_id', 'patient_nbr'], errors='ignore')
df_clean = df_clean[df_clean['gender'].isin(['Male', 'Female'])]

print(f"Cleaned dataset shape: {df_clean.shape}")

### Diagnose Grouping (ICD-9 Code Mapping)
We group the thousands of diagnosis codes into 9 categories to build a robust classifier.

In [ ]:
def map_icd9(code):
    if pd.isna(code) or code == '?':
        return 'Other'
    code = str(code).strip()
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        val = float(code)
        if 390 <= val <= 459 or val == 785: return 'Circulatory'
        elif 460 <= val <= 519 or val == 786: return 'Respiratory'
        elif 520 <= val <= 579 or val == 787: return 'Digestive'
        elif 250 <= val < 251: return 'Diabetes'
        elif 800 <= val <= 999: return 'Injury'
        elif 710 <= val <= 739: return 'Musculoskeletal'
        elif 580 <= val <= 629 or val == 788: return 'Genitourinary'
        elif 140 <= val <= 239: return 'Neoplasms'
        else: return 'Other'
    except:
        return 'Other'

df_clean['diag_1_cat'] = df_clean['diag_1'].apply(map_icd9)
df_clean['diag_2_cat'] = df_clean['diag_2'].apply(map_icd9)
df_clean['diag_3_cat'] = df_clean['diag_3'].apply(map_icd9)
df_clean = df_clean.drop(columns=['diag_1', 'diag_2', 'diag_3'])

# Handle specialties
df_clean['medical_specialty'] = df_clean['medical_specialty'].fillna('Missing')
top_specialties = df_clean['medical_specialty'].value_counts().index[:10]
df_clean['medical_specialty'] = df_clean['medical_specialty'].apply(lambda x: x if x in top_specialties else 'Other')

# Map admissions/discharges
df_clean['admission_type_group'] = df_clean['admission_type_id'].apply(lambda x: 'Emergency' if x in [1, 2, 7] else ('Elective' if x == 3 else 'Other'))
df_clean['admission_source_group'] = df_clean['admission_source_id'].apply(lambda x: 'Emergency Room' if x == 7 else ('Referral' if x in [1, 2] else 'Other'))
df_clean['discharge_disposition_group'] = df_clean['discharge_disposition_id'].apply(lambda x: 'Home' if x == 1 else ('Transfer' if x in [3, 4, 5, 22] else 'Other'))
df_clean = df_clean.drop(columns=['admission_type_id', 'admission_source_id', 'discharge_disposition_id'])

# Retain key columns to make the classification space simpler
numeric_features = ['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
categorical_features = ['race', 'gender', 'medical_specialty', 'diag_1_cat', 'diag_2_cat', 'diag_3_cat', 'admission_type_group', 'admission_source_group', 'discharge_disposition_group', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed', 'metformin', 'insulin']

df_clean = df_clean[numeric_features + categorical_features + ['readmitted']]
print(f"Final preprocessed features shape: {df_clean.shape}")

## 5. Train-Test Split and SMOTE Oversampling
We partition the dataset and oversample the training split using SMOTE to handle class imbalance.

In [ ]:
X = df_clean.drop(columns=['readmitted'])
y = df_clean['readmitted']

# Fill missing categoricals with 'Missing'
for col in categorical_features:
    X[col] = X[col].fillna('Missing')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Build column transformer pipeline
preprocessor = ColumnTransformer( 
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Retrieve column names
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
feature_names = numeric_features + encoded_cat_features

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print(f"Target distribution BEFORE SMOTE:\n{y_train.value_counts(normalize=True)}")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_df, y_train)
print(f"Target distribution AFTER SMOTE:\n{y_train_res.value_counts(normalize=True)}")

## 6. Model Training, Evaluation, and Comparison
We train the 4 required classifiers: Logistic Regression, Decision Tree, Random Forest, and XGBoost, and compare their performance.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, C=0.1),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss')
}

results = []
plt.figure(figsize=(10, 8))

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_df)
    y_prob = model.predict_proba(X_test_df)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend()
plt.show()

comparison_df = pd.DataFrame(results)
print(comparison_df)

## 7. Model Interpretation via SHAP
We construct a SHAP summary plot on the test set using the best-performing model (typically XGBoost or Random Forest).

In [ ]:
# Find best model name by ROC-AUC
best_model_name = comparison_df.sort_values(by="ROC-AUC", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best model selected for explanation: {best_model_name}")

# Explain using SHAP
try:
    if "XGB" in best_model_name or "Random Forest" in best_model_name:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_test_df)
        if isinstance(shap_values, list):
            shap_values = shap_values[1] # RF positive class
        elif len(shap_values.shape) == 3:
            shap_values = shap_values[:, :, 1]
            
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_test_df)
    else:
        print("Running generic SHAP Explainer (can take a minute)...")
        explainer = shap.Explainer(best_model, X_test_df)
        shap_values = explainer(X_test_df)
        shap.plots.beeswarm(shap_values)
except Exception as e:
    print(f"Failed to plot SHAP: {e}")